In [1]:
 import requests, re

In [2]:
import requests
import re
import json
import time


def extract_emails(html):
    """Extract email addresses from HTML using a stricter regex."""
    pattern = r"[\w.+-]+@[\w-]+\.(?:com|org|net|edu|gov|io|co|info)\b"
    return set(re.findall(pattern, html, re.IGNORECASE))


def harvest_emails(url, max_retries=2):
    """Fetch a page and extract emails, with browser-like headers."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    for attempt in range(max_retries):
        try:
            print(f"[*] Attempt {attempt + 1} for {url}")
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()

            html = response.text
            emails = extract_emails(html)
            print(f"[+] Found {len(emails)} email(s)")
            return emails

        except requests.exceptions.HTTPError as e:
            print(f"[!] HTTP error: {e}")
        except requests.exceptions.ConnectionError:
            print("[!] Connection error - check your internet")
        except Exception as e:
            print(f"[!] Error: {e}")

        if attempt < max_retries - 1:
            time.sleep(2)

    return set()


def main():
   
    base = "http://testphp.vulnweb.com"
    test_urls = [
        base,
        f"{base}/guestbook.php",
        f"{base}/contact.php",
        f"{base}/AJAX/index.php",
    ]

    all_results = {}

    for target_url in test_urls:
        print("\n" + "=" * 60)
        print(f"[+] Harvesting emails from: {target_url}")
        print("=" * 60)

        found = harvest_emails(target_url)
        all_results[target_url] = sorted(found)

        if found:
            print(f"\n[+] Found {len(found)} email address(es):\n")
            for email in sorted(found):
                print(f"    - {email}")
        else:
            print("[i] No email addresses found on this page.")

        time.sleep(2)  # be polite between requests

    with open("harvested_emails_all.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print("\n[i] All results saved to harvested_emails_all.json")


if __name__ == "__main__":
    main()


[+] Harvesting emails from: http://testphp.vulnweb.com
[*] Attempt 1 for http://testphp.vulnweb.com
[!] Connection error - check your internet
[*] Attempt 2 for http://testphp.vulnweb.com
[!] Connection error - check your internet
[i] No email addresses found on this page.

[+] Harvesting emails from: http://testphp.vulnweb.com/guestbook.php
[*] Attempt 1 for http://testphp.vulnweb.com/guestbook.php
[!] Connection error - check your internet
[*] Attempt 2 for http://testphp.vulnweb.com/guestbook.php
[!] Connection error - check your internet
[i] No email addresses found on this page.

[+] Harvesting emails from: http://testphp.vulnweb.com/contact.php
[*] Attempt 1 for http://testphp.vulnweb.com/contact.php
[!] Connection error - check your internet
[*] Attempt 2 for http://testphp.vulnweb.com/contact.php
[!] Connection error - check your internet
[i] No email addresses found on this page.

[+] Harvesting emails from: http://testphp.vulnweb.com/AJAX/index.php
[*] Attempt 1 for http://t